#### M-RoPE: 인공지능에게 '공간 감각'을 가르치는 GPS

- 인공지능(LLM)은 기본적으로 글을 읽을 때 단어들이 '어떤 순서'로 오는지 알아야 합니다. "개가 사람을 물었다"와 "사람이 개를 물었다"는 단어는 같지만 순서(위치)가 달라서 뜻이 완전히 다르기 때문입니다.
이렇게 각 단어에 순서표(번호표)를 달아주는 기술을 RoPE(로프)라고 합니다.

- 하지만 이미지를 읽게 되면서 문제가 생겼습니다. 이것을 해결한 것이 바로 M-RoPE(Multimodal RoPE)입니다.

1) 문제: "글은 한 줄이지만, 사진은 평면이잖아!"

- 글(텍스트)의 세계 (1차원): 글은 그냥 한 줄로 쭉 이어집니다. 그래서 번호표를 1번, 2번, 3번... 이렇게 일렬로(1D) 달아주면 끝입니다.

- 사진/영상의 세계 (3차원): 사진은 가로와 세로가 있는 '평면(2D)'입니다. 영상은 거기에 '시간(흐름)'까지 더해진 '입체(3D)'입니다.

- 만약 사진을 그냥 1, 2, 3, 4번으로 한 줄로 썰어서 번호를 매기면, 인공지능은 "1번 조각 바로 아래에 5번 조각이 있다"는 위아래의 공간 감각을 잃어버리게 됩니다.

2) 해결책: M-RoPE "영화관 좌석표(3D 주소)를 도입하자!"

- M-RoPE는 인공지능에게 1차원 번호표 대신, 3차원 GPS 좌표(t, h, w)를 달아주는 기술입니다.

- t (Time, 시간): 몇 번째 장면인가? (영상일 경우)
- h (Height, 세로): 위에서부터 몇 번째 줄인가?
- w (Width, 가로): 왼쪽에서부터 몇 번째 칸인가?

In [ ]:
import torch

def get_m_rope_angles(position_ids_thw, head_dim=64):
    """
    position_ids_thw: [시간(t), 세로(h), 가로(w)] 형태의 3D 위치 좌표
    head_dim: 한 번에 처리할 메모리 총 길이 (기본 64칸)
    """
    # 1. 각각의 위치 좌표를 꺼냅니다. (예: t=1, h=2, w=3)
    t_pos = position_ids_thw[0].float()
    h_pos = position_ids_thw[1].float()
    w_pos = position_ids_thw[2].float()

    # RoPE 란?
    # >> 벡터 2개 차원씩 짝(pair)을 지어서,
    # >> 위치 m에 따라 각도 m_(theta)(주파수, 회전 각도) 만큼 2차원 평면에 회전시킨다.
    # 텍스트 t=0(시간 0), h=위치번호, w=위치번호 >> t, h, w 모두 같다. (1차원 벡터처럼 작동)
    # 이미지 t=frame[i] 프레임 번호, h=행 번호, w=열 번호

    # 2. 기준 주파수(파동의 주기)를 생성합니다.
    # RoPE는 쌍(Pair)으로 작동하므로 64차원이면 32쌍의 주파수가 필요합니다.
    # inv_freq = 주파수
    # i : 차원 인덱스, d : dimension(차원 수)
    # i가 작다(앞 차원) -> theta(회전각) 커진다. -> 빠르게 회전 -> 가까운 위치 구분
    # i가 크다(뒤 차원) -> theta(회전각) 작다. -> 천천히 회전 -> 멀리 떨어진 위치 구분
    # 아날로그 초침 시계로 비유하자면,
    # >> 앞차원(고주파)     초침 : 빠르게 돌아서 1초 단위 구분
    # >> 중간 차원(중주파)  분침 : 중간 속도로 돌아서 1분 단위 구분
    # >> 뒷 차원(저주파)    시침 : 천천히 돌아서 1시간 단위 구분
    # >> 이 세계의 칭(초침, 분침, 시침) 합쳐서 정확한 시간(위치) 표현 가능

    inv_freq = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))

    # 3. M-RoPE의 핵심! 32쌍의 주파수를 용도별로 자릅니다. (24, 20, 20 배분) >> 24 + 20 + 20 = 64
    # >> 보통은 동일한 차원 배분하는데, 시간 정보를 좀 더 중요하게 여겨 24를 배분
    # 절반씩 묶이므로 각각 12쌍, 10쌍, 10쌍이 됩니다.
    t_freq = inv_freq[0 : 12]                 # 시간(t)을 위한 12쌍 (24차원 분량)
    h_freq = inv_freq[12 : 22]                # 세로(h)를 위한 10쌍 (20차원 분량)
    w_freq = inv_freq[22 : 32]                # 가로(w)를 위한 10쌍 (20차원 분량)

    # 4. 각 차원별 회전 각도(Angle = 위치 번호 x 주파수)를 계산합니다.
    # outer(외적, outer product)
    t_angle = torch.outer(t_pos, t_freq)
    h_angle = torch.outer(h_pos, h_freq)
    w_angle = torch.outer(w_pos, w_freq)

    # 5. 세 가지 각도를 옆으로 이어 붙입니다. (Concatenation)
    # 바로 이 부분 때문에 12 + 10 + 10 = 32쌍 (64차원)이 '더해져서' 완성됩니다!
    m_rope_angles = torch.cat([t_angle, h_angle, w_angle], dim=-1)

    return m_rope_angles

# --- 테스트 실행 ---
# 상황: 어떤 사진의 조각이 시간상 1초(t), 세로 2번째(h), 가로 3번째(w)에 있습니다.
dummy_pos = torch.tensor([[1], [2], [3]])

print("입력된 3D 주소:", dummy_pos.view(-1).tolist())

angles = get_m_rope_angles(dummy_pos)
# print(angles)
print("완성된 M-RoPE 각도의 형태:", list(angles.shape))
# 출력결과 설명: [1개의 토큰, 32쌍의 각도] -> 32쌍을 삼각함수(sin, cos)에 넣으면 최종 64차원 벡터 완성!
# 이미지의 1장의 패치 >> 1개 토큰

입력된 3D 주소: [1, 2, 3]
완성된 M-RoPE 각도의 형태: [1, 32]
